In [38]:
import math
from typing import Callable
import numpy as np

### Ejercicio 14.
a. Escriba un programa que utilice el algoritmo del adelgazamiento para generar el numero de eventos y las primeras unidades de tiempo de un proceso de Poisson no homogéneo con función de intensidad en los intervalos indicados.

1.
$$
\lambda(t) = 3 + \frac{4}{t+1}, \quad 0 \le t \le 3
$$

2.
$$
\lambda(t) = (t-2)^2 - 5t + 17, \quad 0 \le t \le 5
$$

3.
$$
\lambda(t) = \left \{
    \begin{array}{cl}

        \frac{t}{2} - 1 & : \ 2 \le t \le 3 \\ \\
        1 - \frac{t}{6} & : \  3 \le t \le 6 \\ \\
        0 & : \ \text{en otro caso}

    \end{array}
\right.
$$

b. Indique una forma de mejorar el algoritmo de adelgazamiento para estos ejemplos usando al menos 3 intervalos.

---

In [39]:
def lamda_t_1(t: float):
    if t < 0 or t > 3:
        raise Exception("Out of range")

    return 3 + 4 / (t + 1)

def lamda_t_2(t: float):
    if t < 0 or t > 5:
        raise Exception("Out of range")

    return (t - 2)**2 - 5 * t + 17

def lamda_t_3(t: float):
    if t < 2 or t > 6:
        return 0
    elif t <= 3:
        return (t / 2) - 1
    else:
        return 1 - t / 6

def poisson_no_homogeneo_adelgazamiento(T: float, lamda_t: Callable[[float], float], lamda_cota: float):
    'lamda_t(t): intensidad, lamda_t(t) <= lamda_cota'

    eventos = []
    t = -math.log(1 - np.random.random()) / lamda_cota

    while t <= T:
        V = np.random.random()
        if V < lamda_t(t) / lamda_cota:
            eventos.append(t)

        t -= math.log(1 - np.random.random()) / lamda_cota

    return eventos

print(poisson_no_homogeneo_adelgazamiento(2, lamda_t_1, 7))
print(poisson_no_homogeneo_adelgazamiento(1, lamda_t_2, 21))
print(poisson_no_homogeneo_adelgazamiento(4, lamda_t_3, 0.5))

[0.07271124017871693, 0.3462913207374886, 0.349262225989353, 0.8101629375569694, 0.8565868712722222, 0.8769618675668793, 1.0628327208655657, 1.858086102357098, 1.8728739189919192, 1.8993863300887321, 1.9283106626780617]
[0.022323779129694233, 0.08098682678756405, 0.08714235409546417, 0.0932474636096809, 0.20115173930959693, 0.22229207633392675, 0.2740474967502212, 0.28289434132007546, 0.3011759569780678, 0.39859159164665714, 0.44305679739894416, 0.4443250076722284, 0.5186084635282392, 0.6765947094195532, 0.7620053519464989, 0.763413939873741, 0.9737572552112487, 0.9777832944016448]
[2.97730125610636, 3.5128045554794225]


In [40]:
def poisson_no_homogeneo_adelgazamiento_mejorado(
    T: float,
    lamda_t: Callable[[float], float],
    interv: list[float],
    lamda_cota: list[float]
):
    """
    Se asume que t siempre comienza de 0, por lo que, a modo de ejemplo:
        lamda_t = lambda t: 2*t + 1
        interv = [2, 4, 6]

        Donde interv representaría los siguientes intervalos:
        I_0 = [0, 2]; I_1 = (2, 4]; I_2 = = (4, 6]

        y podríamos tomar
        lamda_cota = [5, 9, 13]

    Tendríamos que:
        assert len(interv) == len(lamda_cota)
        assert lamda_t(t) <= lamda_cota[j], siendo j el intervalo en el cual se encuentra t
        assert T <= interv[-1]
    """

    if T > max(interv):
        raise ValueError("T > lamda_cota.")
    if len(interv) != len(lamda_cota):
        raise ValueError("Faltan lambdas cotas o intervalos.")
    if interv != sorted(interv):
        raise ValueError("La lista de intervalos debe estar ordenada.")

    j = 0  # recorre subintervalos.

    t = -math.log(1 - np.random.random()) / lamda_cota[j]
    eventos = []

    while t <= T:
        if t <= interv[j]:
            V = np.random.random()
            if V < (lamda_t(t) / lamda_cota[j]):
                eventos.append(t)

            t += -math.log(1 - np.random.random()) / lamda_cota[j]

        else:  # t > interv[j]
            t = interv[j] + (t - interv[j]) * lamda_cota[j] / lamda_cota[j + 1]
            j += 1

    return eventos


print(poisson_no_homogeneo_adelgazamiento_mejorado(2, lamda_t_1, [0.33, 1, 3], [7, 6, 5]))
print(poisson_no_homogeneo_adelgazamiento_mejorado(1, lamda_t_2, [1, 1.3, 3], [21, 11, 3]))
print(poisson_no_homogeneo_adelgazamiento_mejorado(4, lamda_t_3, [2.5, 4.5, 6], [0.25, 0.5, 0.25]))

[0.1529902364793044, 0.25229666484256286, 0.705802059273956, 0.7903321769691092, 1.2021423732442396, 1.2494654376251515, 1.765955694872771, 1.7998323620590455, 1.889753015588199, 1.9844179159575377]
[0.10646444900068135, 0.11156004820955874, 0.22106416252572525, 0.2462774072144566, 0.30890880174460245, 0.41091017169670047, 0.46024580963812944, 0.5359562771230121, 0.5679397993320423, 0.5912239022674511, 0.7265573059660273, 0.8901148565883846, 0.9073704557449342, 0.9189798621170245, 0.955085128723587]
[2.577201066557151]
